# Lecture 10: Text pre-processing and Text Retrieval
In this lecture we'll discuss
- Text-preprocessing techniques such as stemming and Tf-Idf
- Semantic Analysis
- Some techniques for searching a document data set for matches to a query document

Note that **semantic** means "relating to meaning in language or logic." 

In [28]:
import numpy as np
import sklearn
from sklearn.datasets import fetch_20newsgroups 
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.decomposition import NMF
import nltk
import re

In [31]:
# Load the data
from nltk.tokenize import word_tokenize
data= fetch_20newsgroups(remove=('headers', 'footers', 'quotes')).data
print(data[1])

A fair number of brave souls who upgraded their SI clock oscillator have
shared their experiences for this poll. Please send a brief message detailing
your experiences with the procedure. Top speed attained, CPU rated speed,
add on cards and adapters, heat sinks, hour of usage per day, floppy disk
functionality with 800 and 1.4 m floppies are especially requested.

I will be summarizing in the next two days, so please add to the network
knowledge base if you have done the clock upgrade and haven't answered this
poll. Thanks.


In [32]:
# Turn this article into a list of words
Test_Article = word_tokenize(data[1])
print(Test_Article)

['A', 'fair', 'number', 'of', 'brave', 'souls', 'who', 'upgraded', 'their', 'SI', 'clock', 'oscillator', 'have', 'shared', 'their', 'experiences', 'for', 'this', 'poll', '.', 'Please', 'send', 'a', 'brief', 'message', 'detailing', 'your', 'experiences', 'with', 'the', 'procedure', '.', 'Top', 'speed', 'attained', ',', 'CPU', 'rated', 'speed', ',', 'add', 'on', 'cards', 'and', 'adapters', ',', 'heat', 'sinks', ',', 'hour', 'of', 'usage', 'per', 'day', ',', 'floppy', 'disk', 'functionality', 'with', '800', 'and', '1.4', 'm', 'floppies', 'are', 'especially', 'requested', '.', 'I', 'will', 'be', 'summarizing', 'in', 'the', 'next', 'two', 'days', ',', 'so', 'please', 'add', 'to', 'the', 'network', 'knowledge', 'base', 'if', 'you', 'have', 'done', 'the', 'clock', 'upgrade', 'and', 'have', "n't", 'answered', 'this', 'poll', '.', 'Thanks', '.']


## Stop words

- These are commonly used words that have little semantic value.
- We typically remove them before performing any text analysis.
- The definition of a stop word is not entirely rigourous, but different languages have different freely downloadable lists of stop words


In [38]:
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))
#print(stop_words)
Test_Article = [w.lower() for w in Test_Article]
#print(Test_Article)
Trimmed_Article = [w for w in Test_Article if not w in stop_words] 
print(Trimmed_Article)

['fair', 'number', 'brave', 'souls', 'upgraded', 'si', 'clock', 'oscillator', 'shared', 'experiences', 'poll', '.', 'please', 'send', 'brief', 'message', 'detailing', 'experiences', 'procedure', '.', 'top', 'speed', 'attained', ',', 'cpu', 'rated', 'speed', ',', 'add', 'cards', 'adapters', ',', 'heat', 'sinks', ',', 'hour', 'usage', 'per', 'day', ',', 'floppy', 'disk', 'functionality', '800', '1.4', 'floppies', 'especially', 'requested', '.', 'summarizing', 'next', 'two', 'days', ',', 'please', 'add', 'network', 'knowledge', 'base', 'done', 'clock', 'upgrade', "n't", 'answered', 'poll', '.', 'thanks', '.']


In [39]:
Trimmed_Article = [w for w in Trimmed_Article if not w=='.']
Trimmed_Article = [w for w in Trimmed_Article if not w== ',']
Trimmed_Article = [w for w in Trimmed_Article if not w== "n't"]
print(Trimmed_Article)

['fair', 'number', 'brave', 'souls', 'upgraded', 'si', 'clock', 'oscillator', 'shared', 'experiences', 'poll', 'please', 'send', 'brief', 'message', 'detailing', 'experiences', 'procedure', 'top', 'speed', 'attained', 'cpu', 'rated', 'speed', 'add', 'cards', 'adapters', 'heat', 'sinks', 'hour', 'usage', 'per', 'day', 'floppy', 'disk', 'functionality', '800', '1.4', 'floppies', 'especially', 'requested', 'summarizing', 'next', 'two', 'days', 'please', 'add', 'network', 'knowledge', 'base', 'done', 'clock', 'upgrade', 'answered', 'poll', 'thanks']


- We probably also want to replace all upper case letters.
## Stemming
- In English, the same base word can serve different grammatical roles.
- E.g. "swim" can be made into "swimming", "swimmers" and "swims"
- All of these words are the same semantically, that is they relate to the same topic or have the same meaning.
- Stemming is the process of removing the suffixes, or ends of words, in a principled manner.
- E.g. "swimming", "swimmers" and "swims" should all be stemmed to "swims"


In [51]:
#Load the stemmer function
porter = nltk.PorterStemmer()
porter.stem('center')

'center'

In [49]:
# Now lets stem the article
Trimmed_Article  = [porter.stem(w) for w in Trimmed_Article]
print(Trimmed_Article)

['fair', 'number', 'brave', 'soul', 'upgrad', 'si', 'clock', 'oscil', 'share', 'experi', 'poll', 'pleas', 'send', 'brief', 'messag', 'detail', 'experi', 'procedur', 'top', 'speed', 'attain', 'cpu', 'rate', 'speed', 'add', 'card', 'adapt', 'heat', 'sink', 'hour', 'usag', 'per', 'day', 'floppi', 'disk', 'function', '800', '1.4', 'floppi', 'especi', 'request', 'summar', 'next', 'two', 'day', 'pleas', 'add', 'network', 'knowledg', 'base', 'done', 'clock', 'upgrad', 'answer', 'poll', 'thank']


## Discussion
 - Numbers?
 - some stop words may have snuck through

## Vectorizing the corpus
- **Terminology:** a collection of documents is called a *corpus*.
- After stemming and removing stop words from our articles, we would like to **vectorize**
- **Goal** convert a corpus to a matrix $X$ where columns of matrix correspond to articles and rows to key-words.
- **Question** How to choose keywords?
- **Question** How to fill in $X_{ij}$?

### Choosing key-words
- As we've already removed stop-words, the remaining words are likely to be informative.
- Compute the *corpus frequency* of all remaining words.
- Choose the $K$ most frequently occuring words as keywords.

### Term weighting

- Call the keywords $w_1,\ldots, w_{K}$. Also, call the documents $a_{1},\ldots, a_{n}$. 
- Count up the number of times $w_i$ appears in $a_j$.
- If $w_i$ appears in $a_j$ frequently, it likely is telling us something about $a_j$.
- On the other hand, if $w_i$ appears in all $a_j$, it is likely telling us something about the *corpus*, and therefore is not useful in distinguishing articles.
- **Solution:** tf-idf weighting
    - $\text{tf}(w_i,a_j) = \# \text{times $w_i$ appears in $a_j$}$ 
    - $\text{df}(w_i) = \# \text{articles containing $w_i$ at least once}$.
    - $n = \# \text{of documents}$ 
    - $\displaystyle X_{ij} = \text{tf}(w_i,a_j)\log\left(\frac{n}{\text{df}(w_i)}\right)$
- Typically **normalize** columnes after: Replace $\mathbf{x}_i$ with $\mathbf{x}_i/\|\mathbf{x}_i\|_2$
    

## In Practice
This is all handled by the following two lines of code:

In [52]:
from sklearn.feature_extraction.text import TfidfVectorizer 
vectorizer = TfidfVectorizer(max_features=2000, min_df=10, stop_words='english')
# max features = max_number of keywords to consider.
# min_df = potential keywords with document frequency less than 10 will be ignored. This means we may get fewer than
# max_features keywords
X = vectorizer.fit_transform(data)
# Above line actually does the tf-idf conversion
idx_to_word = np.array(vectorizer.get_feature_names())
# Above line will get and store the keywords w_i

In [53]:
for word in idx_to_word:
    print(word)

00
000
01
02
03
04
05
06
0d
0t
10
100
1000
11
12
128
13
130
14
145
15
150
16
17
18
19
1988
1989
1990
1991
1992
1993
1d9
1st
1t
20
200
21
22
23
24
25
250
256
26
27
28
29
2nd
2tm
30
300
31
32
33
34
34u
35
36
37
38
386
39
3d
3l
3rd
3t
40
400
41
42
43
44
45
46
47
48
486
49
4t
50
500
51
52
53
54
55
56
57
58
59
5u
60
600
61
62
63
64
65
66
67
68
6ei
6g
70
71
72
73
74
75
75u
76
77
78
79
7ey
7u
80
800
81
82
84
85
86
88
89
90
91
92
93
95
99
9f
9v
__
a86
ability
able
absolute
absolutely
abuse
ac
accept
accepted
access
according
account
accurate
act
action
actions
active
activities
activity
acts
actual
actually
ad
add
added
addition
additional
address
addresses
administration
admit
advance
advantage
advice
afraid
age
agencies
agency
agents
ago
agree
ah
ahead
aid
aids
air
al
algorithm
alive
allow
allowed
allows
alt
alternative
amendment
america
american
americans
amiga
analysis
andrew
angeles
animals
announced
annual
anonymous
answer
answers
anti
anybody
apartment
app
apparently
appear
appears
appl

told
tom
took
tool
toolkit
tools
topic
tor
toronto
total
totally
town
tq
track
trade
traffic
training
transfer
translation
treatment
trial
tried
troops
trouble
true
truly
trust
truth
try
trying
turbo
turkey
turkish
turks
turn
turned
tv
twice
type
types
typical
uk
understand
understanding
unfortunately
unit
united
universe
university
unix
unless
upgrade
usa
use
used
useful
usenet
user
users
uses
using
usr
usual
usually
uunet
uw
valid
value
values
van
vancouver
variety
various
ve
vehicle
version
versions
vga
video
view
views
villages
virtual
visual
voice
volume
vote
vs
wait
waiting
wall
want
wanted
wants
war
warning
washington
wasn
waste
watch
water
way
ways
weak
weapon
weapons
week
weeks
weight
welcome
went
west
western
white
wide
widget
widgets
wife
willing
win
window
windows
wings
winning
wire
wiring
wish
wm
woman
women
won
wonder
wondering
word
words
work
worked
working
works
world
worry
worse
worth
wouldn
write
writes
writing
written
wrong
wrote
wt
ww
x11
x11r5
xlib
xt
xterm
yeah
ye

## Semantic Analysis
- Even if there are millions of documents $\mathbf{x}_i$ there is likely only a small number of topics $\mathbf{w}_1,\ldots,\mathbf{w}_r$. 
- Recall "semantic" means "meaning", so semantic analysis (also called latent semantic indexing) refers to finding these topics/sentiments $\mathbf{w}_i$.
- Typically do this using SVD (aka PCA) or non-negative matrix factorization (NMF)
- *(In the picture below, their $V$ is our $X$)*

![alt text](NMF.png) 

In [54]:
# apply NMF
nmf = NMF(n_components=20, solver="mu")
H = nmf.fit_transform(X)
W = nmf.components_
 
# print the topics
 
for i, topic in enumerate(W):
 
    print("Topic {}: {}".format(i + 1, ",".join([str(x) for x in idx_to_word[topic.argsort()[-10:]]])))

Topic 1: want,way,say,really,did,right,know,time,people,don
Topic 2: appreciated,info,need,ftp,program,information,software,help,looking,mail
Topic 3: lord,church,christians,christian,believe,faith,christ,bible,jesus,god
Topic 4: nsa,algorithm,public,escrow,government,keys,clipper,encryption,chip,key
Topic 5: hd,cd,floppy,controller,ide,hard,drives,disk,scsi,drive
Topic 6: space,20,price,condition,offer,shipping,10,new,sale,00
Topic 7: using,version,problem,ms,program,running,files,dos,file,windows
Topic 8: teams,win,hockey,play,season,players,year,games,team,game
Topic 9: colorado,article,cc,university,cs,soon,banks,gordon,pitt,edu
Topic 10: jewish,arab,turkish,government,armenians,armenian,jews,israeli,people,israel
Topic 11: memory,color,driver,bus,vga,cards,drivers,monitor,video,card
Topic 12: getting,exactly,exist,work,doesn,new,mean,anybody,know,does
Topic 13: windows,appreciate,bike,wondering,info,tell,mail,hi,advance,thanks
Topic 14: buy,driving,bike,price,miles,good,dealer,eng

## Query Matching
- **Problem:**
    - Given corpus $\mathcal{A} = \{a_1,\ldots, a_n\}$.
    - Suppose a *new document* $q_1$ arrives.  
    - **Goal** return $a_{i}$ which is the best semantic match for $q_1$

- Typically, vectorize $\mathcal{A}$ to $X$ **offline**.
- **In practice** want to return match quickly ($\sim 100 $ microseconds)

### Approach 1
 - Recall documents have been **vectorized** to $\mathbf{x}_1,\ldots,\mathbf{x}_n\in\mathbb{R}^{\#\text{keywords}}$ 
 - Vectorize $q_1$ in the same manner: $q_1 \to \mathbf{y}_i\in\mathbb{R}^{\#\text{keywords}}$
 - Return article which is **closest** to $\mathbf{y}_1$.
 - We can be flexible with how we measure closest:
     - $\ell_2$ or Euclidean distance: $\|\mathbf{y}_1 - \mathbf{x}_i\|$.
     - Cosine distance: $\frac{\mathbf{y}^{\top}\mathbf{x}_i}{\|\mathbf{y}\|_2\|\mathbf{x}_i\|_2}$ 
 - **Question:** When might cosine distance be preferable? 

### Approach 2
- Use the **Semantic Structure**
- Suppose we have a matrix factorization: $X = WH$ (or $X \approx WH$ for NMF) 
- As established earlier, this means that $\mathbf{x}_j = \sum_{i=1}^{r}H_{ij}\mathbf{w}_i$ (or $\mathbf{x}_j \approx \sum_{i=1}^{r}H_{ij}\mathbf{w}_i$
- Let $\mathbf{h}_j\in\mathbb{R}^{\#\text{Topics}}$ be the $j$-th column $H$
- **Idea** Project $\mathbf{y}_1$ to the *topic space* first, then measure distance.